# scLiTr analyses — all timepoints or single timepoint

This notebook supports two analysis modes controlled by the `mode` variable in the **Parameters** cell below:

- `mode = "all"` — runs on all three timepoints combined (HH28 + HH31 + HH35).
- `mode = "per_timepoint"` — subsets to a single timepoint specified by `target_experiment`.

**Before running:** set `mode` and (if `per_timepoint`) `target_experiment` and `leiden_resolution` in the Parameters cell, then run all cells top to bottom.

Output filenames (h5ad saves, CSV exports, plot PDFs) are automatically prefixed with `run_label` so results from different modes do not overwrite each other.

In [ ]:
#Important: cache directories must be set before imports
import os

os.environ["NUMBA_CACHE_DIR"] = "/nemo/lab/briscoej/home/users/boeziog/python_cache/numba"
os.environ["XDG_CACHE_HOME"] = "/nemo/lab/briscoej/home/users/boeziog/python_cache"
os.environ["MPLCONFIGDIR"] = "/nemo/lab/briscoej/home/users/boeziog/python_cache/matplotlib"

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import sclitr as sl
import igraph
sc.set_figure_params(dpi=100)
sns.set_style("ticks")

In [ ]:
# ============================================================
# PARAMETERS — edit these before running
# ============================================================

# Analysis mode
mode = "all"           # "all" = all timepoints combined; "per_timepoint" = single timepoint

# Only used when mode == "per_timepoint"
target_experiment = "HH31"  # options: "HH28", "HH31", "HH35"

# Leiden resolution for clonal clustering
# Recommended: 0.5 for mode="all"; 0.7 for mode="per_timepoint"
leiden_resolution = 0.5

In [ ]:
figures_output = (
    "/nemo/lab/briscoej/home/users/boeziog/"
    "LARRY/LARRY_spinal_cord/Jupyter_notebooks/plots/chicken/scLiTr"
)

os.makedirs(figures_output, exist_ok=True)

def save_plot(plot_name, annotation=None):
    """
    Save the current matplotlib figure as a PDF.
    Filename is prefixed with run_label so all/per_timepoint outputs are kept separate.

    Parameters
    ----------
    plot_name : str
        Base name of the plot (no extension).
    annotation : str or None
        Optional annotation suffix (e.g. 'cell_type', 'lineage_subdivision').
    """
    suffix = f"_{annotation}" if annotation is not None else ""
    filename = f"{run_label}_{plot_name}{suffix}.pdf"
    save_path = os.path.join(figures_output, filename)
    plt.savefig(save_path, format="pdf", bbox_inches="tight")
    print(f"Saved plot to: {save_path}")

# Import adata and check

In [ ]:
adata = sc.read_h5ad(
    "/nemo/lab/briscoej/home/users/boeziog/LARRY/anndata/all_integrated/preprocessing_larry_chicken_clusters.h5ad"
)

In [ ]:
adata

In [ ]:
adata.obs.columns

## Assign a new "experiment" column

In [ ]:
adata.obs["experiment"] = pd.NA

adata.obs.loc[
    adata.obs["sample"].isin(["BOE5073A1", "BOE5073A2", "BOE5073A3", "BOE5073A4"]),
    "experiment"
] = "HH28"

adata.obs.loc[
    adata.obs["sample"].isin([
        "BOE5073A5", "BOE5073A6", "BOE5073A7", "BOE5073A8",
        "BOE5073A9", "BOE5073A10", "BOE5073A11"
    ]),
    "experiment"
] = "HH35"

adata.obs.loc[
    adata.obs["sample"].isin([
        "BOE5073A13", "BOE5073A14", "BOE5073A15", "BOE5073A16",
        "BOE5073A17", "BOE5073A18", "BOE5073A19", "BOE5073A20"
    ]),
    "experiment"
] = "HH31"

adata.obs["experiment"].value_counts(dropna=False)

## Assign a new samples_LARRY column (control vs LARRY)

In [ ]:
adata.obs["samples_LARRY"] = pd.NA

adata.obs.loc[
    adata.obs["sample"].isin([
        "BOE5073A1", "BOE5073A2", "BOE5073A3", "BOE5073A4",
        "BOE5073A6", "BOE5073A7", "BOE5073A8", "BOE5073A9",
        "BOE5073A10", "BOE5073A11", "BOE5073A14", "BOE5073A15",
        "BOE5073A16", "BOE5073A17", "BOE5073A18", "BOE5073A19", "BOE5073A20"
    ]),
    "samples_LARRY"
] = "LARRY"

adata.obs.loc[
    adata.obs["sample"].isin(["BOE5073A5", "BOE5073A13"]),
    "samples_LARRY"
] = "control"

adata.obs["samples_LARRY"].value_counts(dropna=False)

## Assign detected vs non detected

In [ ]:
adata.obs["clone_id_detected"] = pd.NA
adata.obs.loc[
    adata.obs["experiment"].isin(["HH28", "HH31", "HH35"]) &
    adata.obs["clone"].notna(),
    "clone_id_detected"
] = "detected"
adata.obs.loc[
    adata.obs["experiment"].isin(["HH28", "HH31", "HH35"]) &
    adata.obs["clone"].isna(),
    "clone_id_detected"
] = "not_detected"
adata.obs["clone_id_detected"].value_counts(dropna=False)

In [ ]:
# ── Mode-based filtering ──────────────────────────────────────────────
if mode == "per_timepoint":
    adata = adata[adata.obs["experiment"] == target_experiment].copy()
    run_label = target_experiment
    print(f"Subsetting to experiment: {target_experiment}. Remaining cells: {adata.n_obs}")
else:
    run_label = "allsamples"
    print(f"Running on all timepoints combined. Total cells: {adata.n_obs}")

print(f"run_label = '{run_label}'")

# Analyses scLiTr

### Check the biggest clone

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 6))

# Cell types
sc.pl.embedding(
    adata,
    basis="X_umap",
    color="cell_type",
    frameon=False,
    title="Cell type",
    legend_loc="on data",
    legend_fontsize=9,
    legend_fontoutline=2,
    ax=axes[0],
    show=False,
)

# One clone highlighted
sl.pl.clone(
    adata,
    clone_col="clone",
    clone_name=adata.obs["clone"].value_counts().index[0],
    basis="X_umap",
    title="Largest clone",
    kwargs_clone={"color": "experiment"},
    ax=axes[1],
)


fig.tight_layout()

In [ ]:
#check clone size distribution
sl.pl.basic_stats(adata, obs_name="clone", title="Clone size distribution")

In [ ]:
#Subset adata to remove nan clones
adata_clone2vec = adata[adata.obs["clone"].notna()].copy()

In [ ]:
#We retain clones with >=2 cells (median clone size ~4)

sl.tl.clonal_nn(
    adata_clone2vec,
    obs_name="clone",
    use_rep="X_scVI",
    min_size=2,
    tqdm_bar=True,
)

In [ ]:
adata_clone2vec.uns.keys()
adata_clone2vec.obsm.keys()

In [ ]:
adata_clone2vec.obsm["bag-of-clones"].shape

In [ ]:
adata_clone2vec.uns["bag-of-clones_names"][:10]

In [ ]:
clones = sl.tl.clone2vec(
    adata_clone2vec,
    obs_name="clone",
    fill_ct=None,
    device="cpu",
    n_epochs=100,
)

In [ ]:
output_clones_h5ad = f"{run_label}_clones_clone2vec_trained.h5ad"
clones.write_h5ad(output_clones_h5ad)
print(f"Saved clones with clone2vec to {output_clones_h5ad}")

In [ ]:
#Open clones if saved previously
clones = sc.read_h5ad(f"{run_label}_clones_clone2vec_trained.h5ad")
print(f"Loaded clones from {run_label}_clones_clone2vec_trained.h5ad")

In [ ]:
sl.pl.epochs_loss(clones)

In [ ]:
clones

In [ ]:
sc.pp.neighbors(clones, use_rep="clone2vec", n_neighbors=15)
sc.tl.umap(clones)

In [ ]:
sc.pl.umap(clones, frameon=False)

In [ ]:
sc.tl.leiden(clones, resolution=leiden_resolution)

sc.pl.umap(
    clones,
    color="leiden",
    frameon=False,
    title="Clonal clusters",
    legend_loc="on data",
    legend_fontsize=15,
    legend_fontoutline=3,
    show=False
)

save_plot("clonal_clusters_UMAP")
plt.close()

In [ ]:
sl.tl.transfer_clonal_annotation(
    adata,
    clones,
    adata_clone_name="clone",          # column in adata.obs
    adata_obs_name="clonal_cluster",   # NEW column to create in adata.obs
    clones_obs_name="leiden",           # column in clones.obs
    fill_values="NA",
)

In [ ]:
adata.obs["clonal_cluster_plot"] = adata.obs["clonal_cluster"].replace("NA", np.nan)

sc.pl.umap(
    adata,
    color=["clonal_cluster_plot", "cell_type"],
    frameon=False,
    ncols=2,
    show=False
)

#save_plot("cells_UMAP_and_clonal_clusters")
#plt.close()

In [ ]:
adata.obsm["X_umap_2d"] = adata.obsm["X_umap"][:, :2]

In [ ]:
# NOTE: cluster IDs depend on the Leiden run and will differ between
# mode='all' and mode='per_timepoint'. Inspect clones.obs['leiden'].unique()
# after running Leiden above and update this list accordingly.
# Default below matches the allsamples run (7 clusters: 0-6).
# For per_timepoint (resolution=0.7) the allsamples notebook observed 6 clusters (0-5).
clusters = sorted(clones.obs["leiden"].unique().tolist())
n_clusters = len(clusters)

fig, axes = plt.subplots(
    nrows=2,
    ncols=n_clusters,
    figsize=(2.5 * n_clusters, 6),
)

for i, clonal_cluster in enumerate(clusters):
    sl.pl.kde(
        adata,
        basis="X_umap_2d",
        groupby="clonal_cluster",
        group=clonal_cluster,
        title=f"Clonal cluster {clonal_cluster}",
        ax=axes[0, i],
    )

    sl.pl.clone(
        adata,
        clone_col="clonal_cluster",
        clone_name=clonal_cluster,
        basis="X_umap_2d",
        title=f"Clonal cluster {clonal_cluster}",
        s=10,
        ax=axes[1, i],
    )


save_plot("clonal_clusters_density_on_cells_UMAP")

fig.tight_layout()

# Bar plots clonal clusters

## Barplots cell types

In [ ]:
# Subset only cells with assigned clonal clusters (for plotting)
df = adata.obs.loc[
    adata.obs["clonal_cluster"].notna(),
    ["clonal_cluster", "cell_type"]
].copy()

#subset cells and clonal cluster non NA
df = df[
    df["clonal_cluster"].notna() &
    (df["clonal_cluster"] != "NA")
].copy()

# Count cells per (clonal_cluster x cell_type)
counts = (
    df
    .groupby(["clonal_cluster", "cell_type"], observed=True)
    .size()
    .reset_index(name="n_cells")
)

# Normalize within each clonal cluster
counts["fraction"] = (
    counts["n_cells"] /
    counts.groupby("clonal_cluster", observed=True)["n_cells"].transform("sum")
)

In [ ]:
counts

In [ ]:
#choose order
cluster_order = (
    counts
    .groupby("clonal_cluster", observed=True)["n_cells"]
    .sum()
    .sort_values(ascending=False)
    .index
)

In [ ]:
#pivot for plotting

plot_df = (
    counts
    .pivot(index="clonal_cluster", columns="cell_type", values="fraction")
    .fillna(0)
    .loc[cluster_order]
)

In [ ]:
# Clean dataframe
df = adata.obs.loc[
    adata.obs["clonal_cluster"].notna() &
    (adata.obs["clonal_cluster"] != "NA"),
    ["clonal_cluster", "cell_type"]
].copy()

# Count + normalize
counts = (
    df
    .groupby(["clonal_cluster", "cell_type"], observed=True)
    .size()
    .reset_index(name="n_cells")
)

counts["fraction"] = (
    counts["n_cells"] /
    counts.groupby("clonal_cluster", observed=True)["n_cells"].transform("sum")
)

# Order clusters by size
cluster_order = (
    counts
    .groupby("clonal_cluster", observed=True)["n_cells"]
    .sum()
    .sort_values(ascending=False)
    .index
)

# Pivot
plot_df = (
    counts
    .pivot(index="clonal_cluster", columns="cell_type", values="fraction")
    .fillna(0)
    .loc[cluster_order]
)

# Palette
base_colors = [
    "#4E79A7", "#59A14F", "#E15759", "#F28E2B",
    "#76B7B2", "#EDC948", "#B07AA1", "#9C755F",
    "#BAB0AC"
]
palette = (sns.color_palette(base_colors, n_colors=9) * 3)[:27]

# Plot
fig, ax = plt.subplots(figsize=(1.4 * plot_df.shape[0], 6))

plot_df.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    width=0.9,
    color=palette
)

ax.set_ylim(0, 1)
ax.set_ylabel("Fraction of cells")
ax.set_xlabel("Clonal cluster")
ax.set_title("Cell-type composition of clonal clusters")


ax.legend(
    title="Cell type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=8,
    title_fontsize=9,
    ncol=2
)


plt.tight_layout()
plt.show()

### Same cell type bar plots but without general immature neurons

In [ ]:
# ------------------------------------------------------------
# Stacked bar plot of cell-type composition per clonal cluster
# Non-informative cell types are excluded for visualization
# (cells are NOT removed from the AnnData object)
# ------------------------------------------------------------

import seaborn as sns
import matplotlib.pyplot as plt

# 1. Define non-informative cell types to exclude (exact matches only)
excluded_cell_types = {
    "Immature neurons",
    "immature neurons",
    "Unknown",
}

# 2. Subset metadata for plotting ONLY
df_plot = adata.obs.loc[
    adata.obs["clonal_cluster"].notna() &
    adata.obs["cell_type"].notna() &
    ~adata.obs["cell_type"].isin(excluded_cell_types),
    ["clonal_cluster", "cell_type"]
].copy()

# 3. Remove categorical ghost levels (e.g. "NA")
df_plot["clonal_cluster"] = df_plot["clonal_cluster"].astype(str)
df_plot = df_plot[df_plot["clonal_cluster"] != "NA"]

# 4. Count cells per (clonal cluster x cell type)
counts = (
    df_plot
    .groupby(
        ["clonal_cluster", "cell_type"],
        observed=True
    )
    .size()
    .reset_index(name="n_cells")
)

# 5. Convert counts to fractions within each clonal cluster
plot_df = (
    counts
    .groupby(
        ["clonal_cluster", "cell_type"],
        observed=True
    )["n_cells"]
    .sum()
    .groupby(level=0)
    .apply(lambda x: x / x.sum())
    .unstack(fill_value=0)
)

# 6. Drop clonal clusters that are empty after filtering
plot_df = plot_df.loc[plot_df.sum(axis=1) > 0]

# 7. Define publication-friendly color palette
palette = (
    sns.color_palette("tab20", 20)
    + sns.color_palette("tab20b", 20)
    + sns.color_palette("tab20c", 20)
)[:plot_df.shape[1]]

palette = sns.color_palette(palette, desat=0.6)


# 8. Plot stacked bar plot
fig, ax = plt.subplots(figsize=(1.4 * plot_df.shape[0], 6))

plot_df.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=palette,
    width=0.9
)

# Correct x-axis labels (MultiIndex-safe)
ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(
    plot_df.index.get_level_values(0).astype(str),
    rotation=0
)

ax.set_ylim(0, 1)
ax.set_ylabel("Fraction of cells")
ax.set_xlabel("Clonal cluster")
ax.set_title("Cell-type composition of clonal clusters")

# 9. Legend formatting
ax.legend(
    title="Cell type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=8,
    title_fontsize=9,
    ncol=2
)

save_plot("clonal_cluster_barplot", annotation="cell_type")

sns.despine()
plt.tight_layout()
plt.show()

## Same barplot but with lineage subdivisions

In [ ]:
sorted(df_plot["cell_type"].dropna().unique())

In [ ]:
#Add lineage sub to the adata

lineage_subdivision_map = {
    "FP": "Sub_1",
    "V3": "Sub_1",
    "p3": "Sub_1",

    "MN": "Sub_2",
    "OPCs": "Sub_2",
    "V2a": "Sub_2",
    "V2b": "Sub_2",
    "p2": "Sub_2",
    "pMN": "Sub_2",

    "V0": "Sub_3",
    "V1": "Sub_3",
    "dI6": "Sub_3",
    "dp6": "Sub_3",
    "p0": "Sub_3",
    "p1": "Sub_3",
    "Immature V0-dI6-V1": "Sub_3",

    "dI3": "Sub_4",
    "dI4_dILA": "Sub_4",
    "dI5_dILB": "Sub_4",
    "dpL": "Sub_4",

    "RP": "Sub_5",
    "dI1": "Sub_5",
    "dI2": "Sub_5",
    "dp1_2": "Sub_5",
}

adata.obs["lineage_subdivision"] = (
    adata.obs["cell_type"]
    .map(lineage_subdivision_map)
)

In [ ]:
# Subset for plotting only

excluded_cell_types = {
    "Immature neurons",
    "immature neurons",
    "Unknown"
}

df_plot = adata.obs.loc[
    adata.obs["clonal_cluster"].notna() &
    (adata.obs["clonal_cluster"] != "NA") &
    ~adata.obs["cell_type"].isin(excluded_cell_types) &
    adata.obs["lineage_subdivision"].notna(),
    ["clonal_cluster", "lineage_subdivision"]
].copy()

In [ ]:
# Compute fractions

counts = (
    df_plot
    .groupby(["clonal_cluster", "lineage_subdivision"], observed=True)
    .size()
    .reset_index(name="n_cells")
)

plot_df = (
    counts
    .pivot_table(
        index="clonal_cluster",
        columns="lineage_subdivision",
        values="n_cells",
        fill_value=0,
        observed=True
    )
)

plot_df = plot_df.div(plot_df.sum(axis=1), axis=0)

In [ ]:
palette = {
    "Sub_1": "#EDC948",  # yellow
    "Sub_2": "#F28E2B",  # orange
    "Sub_3": "#59A14F",  # green
    "Sub_4": "#4E79A7",  # blue
    "Sub_5": "#B07AA1",  # purple / violet
}

In [ ]:
# NOTE: cluster order below matches the allsamples run.
# When mode='per_timepoint', Leiden cluster IDs will differ — inspect
# clones.obs['leiden'].unique() and update this list accordingly.
# allsamples order: ["4", "1", "0", "2", "3", "5"]
# per_timepoint HH31 order (resolution=0.7): ["2", "0", "3", "1", "4"]
cluster_order_lineage = [str(c) for c in sorted(plot_df.index.astype(str).tolist())]

# Reorder plot_df rows (use auto-sorted order; edit cluster_order_lineage above if desired)
plot_df = plot_df.reindex(cluster_order_lineage)

# Drop any clusters that were not present (just in case)
plot_df = plot_df.dropna(how="all")

In [ ]:
# Plot

fig, ax = plt.subplots(figsize=(1.3 * plot_df.shape[0], 6))

plot_df.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[palette[c] for c in plot_df.columns],
    width=0.9
)

ax.set_ylim(0, 1)
ax.set_ylabel("Fraction of cells")
ax.set_xlabel("Clonal cluster")
ax.set_title("Lineage subdivision composition of clonal clusters")

ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df.index.astype(str), rotation=0)

ax.legend(
    title="Lineage subdivision",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=9,
    title_fontsize=10
)

save_plot("clonal_cluster_barplot", annotation="lineage_subdivisions")

plt.tight_layout()
plt.show()

# Heatmap

In [ ]:
# ============================
# USER CHOICES
# ============================

# Choose annotation for heatmap
ANNOTATION = "lineage_subdivision"
#ANNOTATION = "cell_type"

# Order of clonal clusters on y-axis
# NOTE: update when running per_timepoint — cluster IDs differ between modes.
# allsamples default: ["4", "1", "0", "2", "3", "5"]
# per_timepoint HH31 default: ["2", "0", "3", "1", "4"]
cluster_order = ["4", "1", "0", "2", "3", "5"]  # allsamples default

# Cell-type order (dorsal -> ventral)
cell_type_order = [
    "RP", "dp1_2", "dI1", "dI2", "dI3",
    "dpL", "dI4_dILA", "dI5_dILB",
    "dI6", "dp6",
    "p0", "V0", "p1", "V1",
    "V2a", "V2b", "p2", "OPCs", "pMN", "MN",
    "p3", "V3", "FP",
]

# Lineage subdivision order (reversed)
lineage_sub_order = ["Sub_5", "Sub_4", "Sub_3", "Sub_2", "Sub_1"]

# Categories to exclude from plotting only
excluded_cell_types = {"Immature neurons", "immature neurons", "Unknown"}

In [ ]:
# ----------------------------
# Subset metadata
# ----------------------------
df = adata.obs.loc[
    adata.obs["clonal_cluster"].notna(),
    ["clonal_cluster", ANNOTATION]
].copy()

# Remove non-informative cell types only if using cell_type
if ANNOTATION == "cell_type":
    df = df[~df["cell_type"].isin(excluded_cell_types)]

# ----------------------------
# Count and normalize
# ----------------------------
counts = (
    df
    .groupby(["clonal_cluster", ANNOTATION], observed=True)
    .size()
    .reset_index(name="n_cells")
)

heatmap_df = (
    counts
    .groupby(["clonal_cluster", ANNOTATION], observed=True)["n_cells"]
    .sum()
    .groupby(level=0)
    .apply(lambda x: x / x.sum())
    .unstack(fill_value=0)
)

# ----------------------------
# Reorder clonal clusters
# ----------------------------
heatmap_df = heatmap_df.loc[
    [c for c in cluster_order if c in heatmap_df.index]
]

In [ ]:
if ANNOTATION == "cell_type":
    heatmap_df = heatmap_df.reindex(
        columns=[ct for ct in cell_type_order if ct in heatmap_df.columns],
        fill_value=0
    )

elif ANNOTATION == "lineage_subdivision":
    heatmap_df = heatmap_df.reindex(
        columns=[s for s in lineage_sub_order if s in heatmap_df.columns],
        fill_value=0
    )

if ANNOTATION == "cell_type":
    vmax = 0.2
else:
    vmax = None

In [ ]:
plt.figure(figsize=(0.5 * heatmap_df.shape[1] + 3, 4))

sns.heatmap(
    heatmap_df,
    vmax=vmax,
    cmap="OrRd",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Fraction of cells"}
)

ax = plt.gca()
ax.set_yticks(range(len(heatmap_df.index)))
ax.set_yticklabels(
    heatmap_df.index.get_level_values(0).astype(str),
    rotation=0
)

plt.xlabel(ANNOTATION.replace("_", " ").title())
plt.ylabel("Clonal cluster")
plt.title("Clonal cluster composition")

save_plot("clonal_cluster_heatmap_OrRd", annotation=ANNOTATION)
plt.tight_layout()
plt.show()

In [ ]:
adata

In [ ]:
clones

# Analysis of fate commitment predictors

In [ ]:
# Define progenitor cell types
progenitors = [
    "FP", "RP", "dI6", "dp1_2", "dp6", "dpL",
    "p0", "p1", "p2", "p3", "pMN"
]

# Create early-state column
adata.obs["early_state"] = np.where(
    adata.obs["cell_type"].isin(progenitors),
    "Early",
    "Other"
)

adata.obs["early_state"].value_counts()

In [ ]:
#Restrict to prog in clones

adata_clonal = adata[adata.obs["clone"].notna()].copy()
adata_clonal.obs["early_state"].value_counts()

## Clone2vec model

In [ ]:
#Reimport clones if already computed
#clones = sc.read_h5ad(f"{run_label}_clones_clone2vec_trained.h5ad")

In [ ]:
#Run SHAP on prog in clones

shapdata_c2v = sl.tl.predict_c2v(
    adata_clonal,
    clones,
    clone_col="clone",
    ct_col="early_state",
    use_gpu=False,
    verbose=False,
    limit_ct="Early",
    use_ct=False,
    pseudobulk=True,
)

In [ ]:
output_shap_h5ad = f"{run_label}_shapdata_c2v.h5ad"
shapdata_c2v.write_h5ad(output_shap_h5ad)
print(f"Saved SHAP clone2vec model output to {output_shap_h5ad}")

In [ ]:
#Reimport if already computed
shapdata_c2v = sc.read_h5ad(f"{run_label}_shapdata_c2v.h5ad")

In [ ]:
shapdata_c2v.layers.keys()

In [ ]:
shapdata_c2v.var["average_shap"] = (
    shapdata_c2v.layers["shap"].mean(axis=0).A[0]
)

shapdata_c2v.var["average_shap_normalized"] = (
    shapdata_c2v.var["average_shap"] /
    shapdata_c2v.var["average_shap"].max()
)

In [ ]:
shapdata_c2v.var.sort_values("average_shap", ascending=False).head(20)

### Validate model fit

**Note:** The cells below require the model to have been freshly computed (i.e. `sl.tl.predict_c2v()` run in this session) because `eval_set` is not persisted to h5ad. If you loaded `shapdata_c2v` from file, re-run the `predict_c2v` cell first.

In [ ]:
clones.obs["eval_set"].value_counts()

In [ ]:
#Build predicted clone obj
# Copy clones object
clones_predicted = clones.copy()

# Keep only clones that were predicted
clones_predicted = clones_predicted[
    clones.uns["clone2vec_predicted_names"]
]

# Insert predicted clone2vec coordinates
clones_predicted.obsm["clone2vec"] = clones.uns[
    "clone2vec_predicted"
].copy()

# Keep validation clones only
clones_predicted = clones_predicted[
    clones_predicted.obs["eval_set"] == "validation"
].copy()

In [ ]:
#Build real validation set
clones_validation = clones[
    clones.obs["eval_set"] == "validation"
].copy()

In [ ]:
#UMAP comparison

sc.pp.neighbors(clones_predicted, use_rep="clone2vec", n_neighbors=15)
sc.tl.umap(clones_predicted)

sc.pp.neighbors(clones_validation, use_rep="clone2vec", n_neighbors=15)
sc.tl.umap(clones_validation)

fig, axes = plt.subplots(ncols=2, figsize=(8, 4))

sc.pl.umap(
    clones_predicted,
    ax=axes[0],
    show=False,
    frameon=False,
    title="Predicted clone2vec",
    color="leiden",
    legend_loc="on data"
)

sc.pl.umap(
    clones_validation,
    ax=axes[1],
    show=False,
    frameon=False,
    title="Real clone2vec",
    color="leiden",
    legend_loc="on data"
)

plt.tight_layout()

In [ ]:
#Quantitative correlation per dimension
from scipy.stats import pearsonr

common_clones = clones_predicted.obs_names[
    clones_predicted.obs_names.isin(clones_validation.obs_names)
]

n_dims = clones_validation.obsm["clone2vec"].shape[1]

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(15, 6))

row = 0
col = 0

for i in range(min(10, n_dims)):

    ax = axes[row, col]

    x = clones_predicted[common_clones].obsm["clone2vec"][:, i]
    y = clones_validation[common_clones].obsm["clone2vec"][:, i]

    sns.regplot(
        x=x,
        y=y,
        scatter_kws={"s": 10, "color": "grey"},
        line_kws={"color": sns.color_palette()[3]},
        ax=ax,
    )

    r = np.round(pearsonr(x, y)[0], 3)

    ax.set_title(f"Dim {i} (r = {r})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Real")
    ax.grid(alpha=0.3)

    col += 1
    if col == 5:
        col = 0
        row += 1

plt.tight_layout()

### Shap biological interpretations

In [ ]:
shapdata_c2v.var["average_shap"] = (
    shapdata_c2v.layers["shap"].mean(axis=0).A[0]
)

top_genes = shapdata_c2v.var.sort_values(
    "average_shap",
    ascending=False
)

top_genes.head(30)

In [ ]:
top_genes = shapdata_c2v.var.sort_values(
    "average_shap",
    ascending=False
)

top_genes = top_genes.reset_index().rename(columns={"index": "gene"})

output_top_genes_csv = f"{run_label}_top_genes_shapvalues.csv"
top_genes.to_csv(output_top_genes_csv, index=False)
print(f"Saved to {output_top_genes_csv}")

In [ ]:
c2v_df = pd.DataFrame(
    clones.obsm["clone2vec"],
    index=clones.obs_names
)

c2v_df["cluster"] = clones.obs["leiden"]

cluster_means = c2v_df.groupby("cluster", observed=True).mean()

cluster_means

In [ ]:
cluster_means_no6 = cluster_means.drop(index="6", errors="ignore")

In [ ]:
sns.heatmap(cluster_means_no6, cmap="RdBu_r", center=0)

plt.ylabel("Clonal clusters")
plt.xlabel("Clone2vec dimension")
plt.title("Clonal clusters vs clone2vec dimensions")

save_plot("clonal_cluster_vs_dimension")
plt.tight_layout()
plt.show()

In [ ]:
all_top_genes = []

for dim in range(10):
    layer_name = f"shap_c2v{dim}"

    scores = np.array(
        shapdata_c2v.layers[layer_name].mean(axis=0)
    ).flatten()

    gene_df = shapdata_c2v.var.copy()
    gene_df["score"] = scores
    gene_df["dimension"] = dim
    gene_df["gene"] = gene_df.index

    top_df = gene_df.sort_values("score", ascending=False).head(20)

    all_top_genes.append(
        top_df[["dimension", "gene", "score"]]
    )

all_top_genes_df = pd.concat(all_top_genes, ignore_index=True)

all_top_genes_df.head(30)

In [ ]:
output_top_dim_csv = f"{run_label}_top_genes_per_clone2vec_dimension.csv"
all_top_genes_df.to_csv(output_top_dim_csv, index=False)
print(f"Saved to {output_top_dim_csv}")